## Variant region list:
- load the file of all sequences/oligo names (80215)
- make a list of these with a column for the name, sequence, type (if variant: ALT, if reference: REF, if region: region, control (true/false), gene association (all information about the gene (i.e. name with | and ~RAS ...))
- example table is shown [here](https://docs.google.com/spreadsheets/d/1YfepW_nv14024v8KwveGaJTbcBX6pRKRUohl7jZhltY/edit#gid=0)
### Questions:
- Some rows have ALT_, REF_, what does it mean if it has no REF_ and ALT_ is it a region then?

### Process:
- We identified regions near TSS of genes (we looked for variants within these regions (centered in these regions (100bp from the center)))
- Total number of regions: 80215


In [4]:
# imports 
import pandas as pd 
from Bio import SeqIO
# use the config file in yaml format
import yaml
import os
import re

# read config
config_path = "/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/05_variant_region_list/config/config.yaml"
config_path = "./config/config.yaml"
with open(config_path, 'r') as ymlfile:
    config = yaml.safe_load(ymlfile)

In [5]:
# load the data
design_fasta = '/fast/groups/ag_kircher/MPRA/IGVF_Y1_design/resources/association_data/design_no_duplicates_sequence_and_header.fa'
design_fasta = 'resources/design_no_duplicates_sequence_and_header.fa'

# read the fasta file with the sequences and prepare a tsv with header and sequence using biopython

records = list(SeqIO.parse(design_fasta, "fasta"))
design_df = pd.DataFrame(columns=['header', 'sequence'])
header = [] 
sequence = []
for record in records:
    header.append(record.id)
    sequence.append(str(record.seq))

design_df['header'] = header
design_df['sequence'] = sequence

# label is the string in front of the first ":"
design_df['label'] = design_df['header'].str.split(':').str[0]
design_df


    

,header,sequence,label
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,cardiac_neuro_cava_random
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,cardiac_neuro_cava_random
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,cardiac_neuro_cava_random
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,cardiac_neuro_cava_random
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,cardiac_neuro_cava_random
...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,MK
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,MK
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,MK
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,MK


#### Short cut ideas:
- filter all header with variants: |(last pipe)<num>-<num>
  - Pattern ? if variant is there there is ALT in the beginning
- Find variant pattern -> its variant
- Find reference pattern -> its reference
- Find nothing -> its region
- check the numbers

In [86]:

def check_variant(header):
    """Checks if a header is a variant header"""
    # get the last part of the header
    last_part = header.split('|')[-1]
    # check if the last part if it matches the regex [\d]+-[\d]+
    if re.match(r'[\dA-Z]+-[\d]+', last_part): # is sufficient, because all these headers have ALT in their name
        return True
    else:
        return False

def check_reference(header):
    """Checks if a header is a reference header"""
    # check after the label if REF_ is in the header
    non_label_header = header.split(':')[1]
    if "REF_" in non_label_header.split('|')[0]:
        return True
    else:
        return False

def check_region(header):
    """
    Checks if a header is a region header
    A region header is here defined as a header without ALT_ or REF_ after the first ":" 
    """
    # check after the label if REF_ is in the header
    non_label_header = header.split(':')[1]
    if "REF_" in non_label_header.split('|')[0]:
        return False
    elif "ALT_" in non_label_header.split('|')[0]:
        return False
    else:
        return True

def check_region_variant_reference_numbers(header_list):
    """Checks if a header is a variant, reference or region header"""
    ref_counter = 0
    var_counter = 0
    region_counter = 0
    unknown_counter = 0
    for header in header_list:
        if header.split(':')[0] != 'cardiac_neuro_cava_random':
            continue
        if check_variant(header):
            var_counter += 1
            header_type = 'ALT'
        elif check_reference(header):
                ref_counter += 1
                header_type = 'REF'
        elif check_region(header):    
            region_counter += 1
            header_type = 'region'
        else:
            header_tpye = 'unknown'
            unknown_counter += 1
            print(f'Found unknown header: {header}')
    return ref_counter, var_counter, region_counter

def create_fasta_df_from_one_line_sequence_fasta(fasta_path, filter_cardiac=False):
    """Creates a dataframe with header and sequence from a fasta file which has one line per sequence"""
    records = list(SeqIO.parse(fasta_path, "fasta"))
    design_df = pd.DataFrame(columns=['header', 'sequence'])
    header = [] 
    sequence = []
    for record in records:
        header.append(record.id)
        sequence.append(str(record.seq))

    design_df['header'] = header
    design_df['sequence'] = sequence

    if filter_cardiac:
        design_df['label'] = design_df['header'].str.split(':').str[0]
        design_df = design_df[design_df['label'] == 'cardiac_neuro_cava_random']

    return design_df

def get_gene_name_from_header(header):
    """Identify the gene name (e.g. MYH6) from the header: sequence after first ":" and before first "|" then crop the sequence after the first "_" if it exists"""
    gene_name = header.split(':')[1].split('|')[0]
    if '_' in gene_name:
        gene_name = gene_name.split('_')[1]
    return gene_name

def write_variants_fasta(design_fasta_path, fasta_out_directory):
    """Write all variants to a fasta file with the header and sequence"""
    design_df = create_fasta_df_from_one_line_sequence_fasta(design_fasta_path, filter_cardiac=False)
    design_df['is_variant'] = design_df['header'].apply(check_variant)
    design_df = design_df[design_df['is_variant'] == True]
    # write the fasta file
    output_path = os.path.join(fasta_out_directory, f'identified_variants_{design_df.shape[0]}.fa')
    with open(output_path, 'w') as f:
        for index, row in design_df.iterrows():
            f.write('>' + row['header'] + '\n' + row['sequence'] + '\n')
    return design_df

def write_region_fasta(design_fasta_path, fasta_out_directory):
    """Write all regions to a fasta file with the header and sequence"""
    design_df = create_fasta_df_from_one_line_sequence_fasta(design_fasta_path, filter_cardiac=True)
    design_df['is_region'] = design_df['header'].apply(check_region)
    design_df = design_df[design_df['is_region'] == True]
    # write the fasta file
    output_path = os.path.join(fasta_out_directory, f'identified_regions_{design_df.shape[0]}.fa')
    with open(output_path, 'w') as f:
        for index, row in design_df.iterrows():
            f.write('>' + row['header'] + '\n' + row['sequence'] + '\n')
    return design_df

def write_reference_fasta(design_fasta_path, fasta_out_directory):
    """Write all references to a fasta file with the header and sequence"""
    design_df = create_fasta_df_from_one_line_sequence_fasta(design_fasta_path, filter_cardiac=True)
    design_df['is_reference'] = design_df['header'].apply(check_reference)
    design_df = design_df[design_df['is_reference'] == True]
    # write the fasta file
    output_path = os.path.join(fasta_out_directory, f'identified_references_{design_df.shape[0]}.fa')
    with open(output_path, 'w') as f:
        for index, row in design_df.iterrows():
            f.write('>' + row['header'] + '\n' + row['sequence'] + '\n')
    return design_df



In [66]:
# iterate all headers and get the gene name of interest
gene_names = set()
not_cardiac = 0
for hdr in header:
    if hdr.split(':')[0] != 'cardiac_neuro_cava_random':
        not_cardiac += 1
        continue
    # put gene name into set
    gene_name = get_gene_name_from_header(hdr)
    gene_names.add(gene_name)

# check the number of the gene_name set
print(f'Number of unique "assiciated" genes: {len(gene_names)}')  # => Found all (same number as in summary presentation) 525 "associated" genes

Number of unique "assiciated" genes: 525


In [46]:
print(not_cardiac)
num_cardiac = 80215 - 6275
print(num_cardiac)

asdf = 18582 + 46458 + 8900
asdf

6275
73940


73940

In [87]:

ref_counter, var_counter, region_counter = check_region_variant_reference_numbers(header)
print(ref_counter, var_counter, region_counter) # 18582 + 46458 + 8900 = 73940
# we want: 28000 cCREs we got 8900
design_fasta_file = 'resources/design_no_duplicates_sequence_and_header.fa'
output_fasta_directory = 'resources/'
write_variants_fasta(design_fasta_file, output_fasta_directory)
write_reference_fasta(design_fasta_file, output_fasta_directory)
write_region_fasta(design_fasta_file, output_fasta_directory)
# Problem: we are not sure about the exact numbers and the formats might be different
  # - how do I check if the number of variants (currently 46458) is correct?

18582 46458 8900


,header,sequence,label,is_region
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,cardiac_neuro_cava_random,True
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,cardiac_neuro_cava_random,True
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,cardiac_neuro_cava_random,True
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,cardiac_neuro_cava_random,True
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,cardiac_neuro_cava_random,True
...,...,...,...,...
8895,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTTTGGACTCTGGGCTGCTCAGAGGCTGCCTTG...,cardiac_neuro_cava_random,True
8896,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTAGAGCCCTGGGGAACGCCATGAGCCCTCAGG...,cardiac_neuro_cava_random,True
8897,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTTTGGGACTTAAACCCCAGCCTCCCCCGTCCA...,cardiac_neuro_cava_random,True
8898,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTAGCCCATAATTTATTGATTTTTTAAAATTTG...,cardiac_neuro_cava_random,True


In [41]:
# read all headers and check for the regex pattern of variant info after the last pipe ("|")
# count the number of found variant patterns
# if the pattern is found, check if it has ALT_ after the first ":"
# count the number of found variant pattersn with ALT_
var_count = 0
var_count_alt = 0
for hdr in header:
    if hdr.split(':')[0] != 'cardiac_neuro_cava_random':
        continue
    if re.match('ALT_', hdr.split(':')[1]):
        # if it matches, count it
        var_count_alt += 1
        # print(hdr)
        # break
        # get the last part of the header
        last_part = hdr.split('|')[-1]
        # check if the last part if it matches the regex [\d]+-[\d]+
        if re.match(r'[\dA-Z]+-[\d]+', last_part):
            var_count += 1
        else:
            print(hdr)
        
# check if the number of variants is the same as the number of variants with ALT_
if var_count == var_count_alt:
    print("All variants have ALT_ in the header")

All variants have ALT_ in the header


In [6]:
# list of all unique labels
unique_labels = design_df['label'].unique().tolist()
unique_labels

['cardiac_neuro_cava_random',
 'GC_Atrial_fib',
 'GC_Liang',
 'GC_Selvarajan',
 'GC_Mohlke',
 'GC_Kircher',
 'GC_Mendelian_variants',
 'C_positive_heart_CAD',
 'GC_Cort_Chengyu',
 'GC_GABA_Chengyu',
 'GC_Glut_Chengyu',
 'GC_Hon',
 'GC_Vista',
 'GC_DNase_positive',
 'GC_DNase_negative_brain',
 'GC_DNase_negative_blood',
 'C_negative_heart_MK',
 'C_negative_neuron_MK',
 'C_negative_neuron_NP',
 'C_positive_heart_MK',
 'C_positive_neuron_CD',
 'C_positive_neuron_MK',
 'C_positive_neuron_NP',
 'C_positive_heart_AB',
 'C_SLEA',
 'GC_DNase_positive_shuffeled',
 'GC_DNase_negative_brain_shuffeled',
 'GC_DNase_negative_blood_shuffeled',
 'MK']

In [7]:
# investigate the variants
for i in range(40, 48):
    print(design_df['header'].tolist()[i])


cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779571_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779574_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779580_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779583_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779643_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779653_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779655_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779714_fwd_tile1-1


In [8]:
for lable in unique_labels:
    if lable == 'cardiac_neuro_cava_random':
        continue
    df = design_df[design_df['label'] == lable]
    print(lable)
    print(df['header'].str.count('\|').value_counts())

GC_Atrial_fib
header
2    34
4    11
Name: count, dtype: int64
GC_Liang
header
1    16
Name: count, dtype: int64
GC_Selvarajan
header
1    241
2    105
4     10
3      8
Name: count, dtype: int64
GC_Mohlke
header
4     26
12     5
8      3
Name: count, dtype: int64
GC_Kircher
header
400    203
Name: count, dtype: int64
GC_Mendelian_variants
header
2    161
1     48
Name: count, dtype: int64
C_positive_heart_CAD
header
0    97
Name: count, dtype: int64
GC_Cort_Chengyu
header
3    184
2      1
Name: count, dtype: int64
GC_GABA_Chengyu
header
3    85
Name: count, dtype: int64
GC_Glut_Chengyu
header
3    40
Name: count, dtype: int64
GC_Hon
header
1    6
Name: count, dtype: int64
GC_Vista
header
1    256
Name: count, dtype: int64
GC_DNase_positive
header
0    41
Name: count, dtype: int64
GC_DNase_negative_brain
header
0    15
Name: count, dtype: int64
GC_DNase_negative_blood
header
0    15
Name: count, dtype: int64
C_negative_heart_MK
header
0    243
Name: count, dtype: int64
C_negative_neu

<>:6: SyntaxWarning: invalid escape sequence '\|'
<>:6: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_18448/1820163410.py:6: SyntaxWarning: invalid escape sequence '\|'
  print(df['header'].str.count('\|').value_counts())


In [9]:
# filter for the rows with "cardiac_neuro_cava_random" in label column
cardiac_neuro_cava_random = design_df[design_df['label'] == 'cardiac_neuro_cava_random']

# do all of these rows have 2 pipes in the header?
cardiac_neuro_cava_random['header'].str.count('\|').value_counts()


# header
# 5     45082
# 2     27141
# 8       596
# 7       471
# 4       341
# 10      30


<>:5: SyntaxWarning: invalid escape sequence '\|'
<>:5: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_18448/3553961298.py:5: SyntaxWarning: invalid escape sequence '\|'
  cardiac_neuro_cava_random['header'].str.count('\|').value_counts()


header
5     45082
2     27141
8       596
7       471
4       341
10      309
Name: count, dtype: int64

#### 2 pipe example
- 18340 REF
- 27141 all (if no REF_ or ALT_ then it is a region)
- example 
    - cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778476_fwd_tile1-1
    - cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778534_fwd_tile1-1
- format: <label> : <sequence_type> _ <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info>


In [27]:
# show me an example with threshold pipes in the header
num_pipes = 2
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes]['header'].tolist()[0]
# 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778476_fwd_tile1-1' # gene name | ensembl id | enhancer/encodeID_tile id

# does a row with 2 pipes in the header have ALT in the header?
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes]['header'].str.contains('ALT_').value_counts() # no alt there
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes]['header'].str.contains('REF_').value_counts() # no ref there # 18340 REF

# investigate the rows with 2 pipes in the header and ALT in the header
card_cava_2_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes]
print(card_cava_2_pipes.shape) # (27141, 3)
# card_cava_2_pipes['header'].str.contains('ALT')]['header'].tolist()[0]
card_cava_2_pipes[card_cava_2_pipes['header'].str.contains('REF_')]['header'].tolist()[10] # 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778534_fwd_tile1-1'

(27141, 5)


'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778534_fwd_tile1-1'

##### split the dataframe 2 pipe 


In [30]:
# 1: pre_header column: header without the label
# 2: tile info column: tile info (split by "_" and take the last element) + remove this from the pre_header column
# 3: strand column: strand info (split by "_" and take again the last element) + remove this from the pre_header column
# 4: gene_name + ensembl_id + enhancer_id column: split the pre_header column by "|" and take the first 3 elements (expand = True)
# format: <label> : <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info>

# split the header column by ":" and take the second element
card_cava_2_pipes['pre_header'] = card_cava_2_pipes['header'].str.split(':').str[1:].str.join(':')
# tile info: split the pre_header column by "_" and take the last element
card_cava_2_pipes['tile_info'] = card_cava_2_pipes['pre_header'].str.split('_').str[-1]
# remove the tile info from the pre_header column
card_cava_2_pipes['pre_header'] = card_cava_2_pipes['pre_header'].str.split('_').str[:-1].str.join('_')
# strand info: split the pre_header column by "_" and take the last element
card_cava_2_pipes['strand'] = card_cava_2_pipes['pre_header'].str.split('_').str[-1]
# remove the strand info from the pre_header column
card_cava_2_pipes['pre_header'] = card_cava_2_pipes['pre_header'].str.split('_').str[:-1].str.join('_')
# gene_name + ensembl_id + enhancer_id: split the pre_header column by "|" and take the first 3 elements (expand = True)
card_cava_2_pipes[['pre_gene_name', 'ensembl_id', 'enhancer_id']] = card_cava_2_pipes['pre_header'].str.split('|', expand = True)
# remove the pre_header column
card_cava_2_pipes.drop(columns = ['pre_header'], inplace = True)
# sequence_type: put "region" if pre_gene_name does not contain "_" otherwise split the pre_gene_name column by "_" and take the first element
card_cava_2_pipes['sequence_type'] = card_cava_2_pipes['pre_gene_name'].apply(lambda x: 'region' if '_' not in x else x.split('_')[0])
# add gene_name column: split the pre_gene_name column by "_" and take the second element but only if "_" exists in the pre_gene_name column
card_cava_2_pipes['gene_name'] = card_cava_2_pipes['pre_gene_name'].apply(lambda x: x.split('_')[1] if '_' in x else x)
# drop the pre_gene_name column
card_cava_2_pipes.drop(columns = ['pre_gene_name'], inplace = True)

card_cava_2_pipes

/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-24/ipykernel_860954/2199821034.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  card_cava_2_pipes['pre_header'] = card_cava_2_pipes['header'].str.split(':').str[1:].str.join(':')
/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-24/ipykernel_860954/2199821034.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  card_cava_2_pipes['tile_info'] = card_cava_2_pipes['pre_header'].str.split('_').str[-1]
/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-24/ipykernel_8

,header,sequence,label,tile_info,strand,ensembl_id,enhancer_id,sequence_type,gene_name
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,cardiac_neuro_cava_random,tile1-1,fwd,ENSG00000157933.11,EH38E2778476,region,SKI
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,cardiac_neuro_cava_random,tile1-1,fwd,ENSG00000157933.11,EH38E2778477,region,SKI
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,cardiac_neuro_cava_random,tile1-1,fwd,ENSG00000157933.11,EH38E2778478,region,SKI
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,cardiac_neuro_cava_random,tile1-1,fwd,ENSG00000157933.11,EH38E2778480,region,SKI
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,cardiac_neuro_cava_random,tile1-1,fwd,ENSG00000157933.11,EH38E2778484,region,SKI
...,...,...,...,...,...,...,...,...,...
27477,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,AGGACCGGATCAACTACCCCACTGCTGCACCAGATTGAGCTGGAGA...,cardiac_neuro_cava_random,tile1-1,rev,ENSG00000160211.20,EH38E3949715,REF,G6PD
27478,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,AGGACCGGATCAACTTTGCTGAGTAGTATCCGTTGTATGAATGCAC...,cardiac_neuro_cava_random,tile1-1,rev,ENSG00000160211.20,EH38E3949725,REF,G6PD
27479,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,AGGACCGGATCAACTGGAGCTCTGCCTCACCCCACCTGGCCCCAAT...,cardiac_neuro_cava_random,tile1-1,rev,ENSG00000160211.20,EH38E3949733,REF,G6PD
27480,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,AGGACCGGATCAACTATGTCTGAATTCACCTCCAAATAATGGGAAA...,cardiac_neuro_cava_random,tile1-1,rev,ENSG00000160211.20,EH38E2774396,REF,G6PD


#### 4 pipe example (what group is this, mohan?)
- 242 REF / 99 (region)
- example: 
    - cardiac_neuro_cava_random:CSDE1|ENSG00000009307.17|EH38E1378377~NRAS|ENSG00000213281.5|EH38E1378377_rev_tile1-1
    - cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1
- format: `<label> : [<sequence_type>_]*<gene name> | <ensembl id> | <enhancer/encode id> ~ <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info>`

In [36]:
# show me an example with threshold pipes in the header
num_pipes = 4
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes]['header'].tolist()[0]
# 'cardiac_neuro_cava_random:CSDE1|ENSG00000009307.17|EH38E1378377~NRAS|ENSG00000213281.5|EH38E1378377_rev_tile1-1'

card_cava_4_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes] # 341 rows
print(len(card_cava_4_pipes)) # 341
# do all these rows have ALT in the header?
card_cava_4_pipes['header'].str.contains('ALT_').value_counts() # no, no alt 
card_cava_4_pipes['header'].str.contains('REF_').value_counts() # True, 242 

# check the header of something containing REF_
card_cava_4_pipes[card_cava_4_pipes['header'].str.contains('REF_')]['header'].tolist()[0] # 'cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1'

# check the header of something not containing REF_
card_cava_4_pipes[~card_cava_4_pipes['header'].str.contains('REF_')]['header'].tolist()[0] # 'cardiac_neuro_cava_random:CSDE1|ENSG00000009307.17|EH38E1378377~NRAS|ENSG00000213281.5|EH38E1378377_rev_tile1-1'

# check how many rows have ~ in the header
card_cava_4_pipes['header'].str.count('~').value_counts() # all have just one "~" 341

# check if all rows have 3 "_" in the header
card_cava_4_pipes['header'].str.count('_').value_counts() # all have 3 "_" 341

341


header
6    242
5     99
Name: count, dtype: int64

In [40]:
# format: `<label> : [<sequence_type>_]*<gene name> | <ensembl id> | <enhancer/encode id> ~ <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info>`
# 1: pre_header column: header without the label
# 2: "additional_gene_association" column: split the pre_header column by "~" and take the second element
# 3: tile info column: tile info (split "additional_gene_association" by "_" and take the last element) + remove this from the column
# 4: strand info column: strand info (split "additional_gene_association" by "_" and take again the last element) + remove this from the column
# 5: remove the "additional_gene_association" column from the pre_header column (split by "~" and take the first element)
# 6: pre_gene_name + ensembl_id + enhancer_id: split the pre_header column by "|" and take the first 3 elements (expand = True)
# 7: remove the pre_header column
# 8: sequence_type: put "region" if pre_gene_name does not contain "_" otherwise split the pre_gene_name column by "_" and take the first element

# split the header column by ":" and take the second element
card_cava_4_pipes['pre_header'] = card_cava_4_pipes['header'].str.split(':').str[1:].str.join(':')
# additional_gene_association: split the pre_header column by "~" and take the second element
card_cava_4_pipes['additional_gene_association'] = card_cava_4_pipes['pre_header'].str.split('~').str[1]
# tile info: split the additional_gene_association column by "_" and take the last element
card_cava_4_pipes['tile_info'] = card_cava_4_pipes['additional_gene_association'].str.split('_').str[-1]
# remove the tile info from the additional_gene_association column
card_cava_4_pipes['additional_gene_association'] = card_cava_4_pipes['additional_gene_association'].str.split('_').str[:-1].str.join('_')
# strand info: split the additional_gene_association column by "_" and take the last element
card_cava_4_pipes['strand'] = card_cava_4_pipes['additional_gene_association'].str.split('_').str[-1]
# remove the strand info from the additional_gene_association column
card_cava_4_pipes['additional_gene_association'] = card_cava_4_pipes['additional_gene_association'].str.split('_').str[:-1].str.join('_')
# remove the additional_gene_association column from the pre_header column
card_cava_4_pipes['pre_header'] = card_cava_4_pipes['pre_header'].str.split('~').str[0]
# pre_gene_name + ensembl_id + enhancer_id: split the pre_header column by "|" and take the first 3 elements (expand = True)
card_cava_4_pipes[['pre_gene_name', 'ensembl_id', 'enhancer_id']] = card_cava_4_pipes['pre_header'].str.split('|', expand = True)
# remove the pre_header column
card_cava_4_pipes.drop(columns = ['pre_header'], inplace = True)
# sequence_type: put "region" if pre_gene_name does not contain "_" otherwise split the pre_gene_name column by "_" and take the first element
card_cava_4_pipes['sequence_type'] = card_cava_4_pipes['pre_gene_name'].apply(lambda x: 'region' if '_' not in x else x.split('_')[0])
# add gene_name column: split the pre_gene_name column by "_" and take the second element but only if "_" exists in the pre_gene_name column
card_cava_4_pipes['gene_name'] = card_cava_4_pipes['pre_gene_name'].apply(lambda x: x.split('_')[1] if '_' in x else x)
# drop the pre_gene_name column
card_cava_4_pipes.drop(columns = ['pre_gene_name'], inplace = True)

card_cava_4_pipes


/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-24/ipykernel_860954/813458402.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  card_cava_4_pipes['pre_header'] = card_cava_4_pipes['header'].str.split(':').str[1:].str.join(':')
/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-24/ipykernel_860954/813458402.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  card_cava_4_pipes['additional_gene_association'] = card_cava_4_pipes['pre_header'].str.split('~').str[1]
/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cp

,header,sequence,label,tile_info,additional_gene_association,strand,ensembl_id,enhancer_id,sequence_type,gene_name
708,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTAAGGAAGGGAGGGAGGGAGGGAGCGATCCCT...,cardiac_neuro_cava_random,tile1-1,NRAS|ENSG00000213281.5|EH38E1378377,rev,ENSG00000009307.17,EH38E1378377,region,CSDE1
709,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTGGACCACCTCCACCAACTGTCAGCTCACATC...,cardiac_neuro_cava_random,tile1-1,NRAS|ENSG00000213281.5|EH38E2832502,rev,ENSG00000009307.17,EH38E2832502,region,CSDE1
710,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTACTGTCCTTCCTGAGGCCTCCAGCATTATTG...,cardiac_neuro_cava_random,tile1-1,NRAS|ENSG00000213281.5|EH38E2832508,rev,ENSG00000009307.17,EH38E2832508,region,CSDE1
711,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTTTACTTATTTATTTATTTTTTGAGACAGGGT...,cardiac_neuro_cava_random,tile1-1,NRAS|ENSG00000213281.5|EH38E2832513,rev,ENSG00000009307.17,EH38E2832513,region,CSDE1
712,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTACCCCATAGTATGCCTCCCTCCCTCTCTCCT...,cardiac_neuro_cava_random,tile1-1,NRAS|ENSG00000213281.5|EH38E1378387,rev,ENSG00000009307.17,EH38E1378387,region,CSDE1
...,...,...,...,...,...,...,...,...,...,...
27156,cardiac_neuro_cava_random:REF_IQSEC2|ENSG00000...,AGGACCGGATCAACTAATCCTTTGAGAGTTATTAATGAGATAGACT...,cardiac_neuro_cava_random,tile1-1,KDM5C|ENSG00000126012.13|EH38E3934846,rev,ENSG00000124313.18,EH38E3934846,REF,IQSEC2
27157,cardiac_neuro_cava_random:REF_IQSEC2|ENSG00000...,AGGACCGGATCAACTATGGAGAAAGCAAGTTGCAGAACAGGGTTTA...,cardiac_neuro_cava_random,tile1-1,KDM5C|ENSG00000126012.13|EH38E2755266,rev,ENSG00000124313.18,EH38E2755266,REF,IQSEC2
27166,cardiac_neuro_cava_random:REF_IQSEC2|ENSG00000...,AGGACCGGATCAACTCTTCTGGGAGTCAGTCTTGCATCTTGTCCCT...,cardiac_neuro_cava_random,tile1-1,SMC1A|ENSG00000072501.19|EH38E3934894,rev,ENSG00000124313.18,EH38E3934894,REF,IQSEC2
27167,cardiac_neuro_cava_random:REF_IQSEC2|ENSG00000...,AGGACCGGATCAACTGAAGGGGCGGCGAATATGCATTGAAGGAATC...,cardiac_neuro_cava_random,tile1-1,SMC1A|ENSG00000072501.19|EH38E3934896,rev,ENSG00000124313.18,EH38E3934896,REF,IQSEC2


enhancer id column; other genes with same enhancer
- mohan idea: 2D array or 3D array (gene column will have multiple genes)
- EH38E1378377~NRAS|ensemble id ... 
- fwd: (+ strand)
- rev: (- strand)
- gene name does not implicitly tell you the enhancer id (e.g. JUP|ENSG00000173801.17|EH38E3223379)

#### 5 pipe examples are all alt / variants (45082)
- 45082 alt
- example: 
    - cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778471|1-2179591-T-C
    - cardiac_neuro_cava_random:ALT_ST3GAL3|ENSG00000126091.21|EH38E1342813_fwd_tile1-1_ST3GAL3|ENSG00000126091.21|EH38E1342813|1-43835787-G-A
    - cardiac_neuro_cava_random:ALT_IGLV3-25|ENSG00000211659.2|EH38E3470175_fwd_tile1-1_IGLV3-25|ENSG00000211659.2|EH38E3470175|22-22639049-T-C
- format: `<label> : <sequence_type (all ALT)> _ <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info> _ <gene name> | <ensembl id> | <enhancer/encode id> | <variant info>`

In [48]:
num_pipes = 5
# show me an example with 5 pipes in the header
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes]['header'].tolist()[2323]
# idx 0: cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778471|1-2179591-T-C
# idx 10: cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778513_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778513|1-2203222-G-A
# idx 24: cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778546_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778546|1-2215527-T-C
# idx 2400: cardiac_neuro_cava_random:ALT_POMGNT1|ENSG00000085998.15|EH38E2809115_rev_tile1-1_POMGNT1|ENSG00000085998.15|EH38E2809115|1-46191391-T-C
# idx 2323: cardiac_neuro_cava_random:ALT_ST3GAL3|ENSG00000126091.21|EH38E1342813_fwd_tile1-1_ST3GAL3|ENSG00000126091.21|EH38E1342813|1-43835787-G-A


# investigate and find pattern
card_cava_5_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes] 
# does all of these rows have ALT_ after the first ":" in the header?
card_cava_5_pipes['header'].str.split(':').str[1].str.contains('ALT_').value_counts() # yes => all with 5 pipes have ALT_ in name (45082)
# card_cava_5_pipes['header'].str.split(':').str[1].str.contains('REF_').value_counts() #

# # do all of these rows end with the regex /-[A-Z]*-[A-Z]*/
# card_cava_5_pipes['header'].str.extract(r'(-[A-Z]*-[A-Z]*)$')[0].value_counts() # yes => all with 5 pipes have ALT_ in name

# check if all rows have the same number of "-"
card_cava_5_pipes['header'].str.count('-').value_counts() # 44237 have 4 and 845 have 6
# check what the 6 "-" rows look like
card_cava_5_pipes[card_cava_5_pipes['header'].str.count('-') == 6]['header'].tolist()[0] # cardiac_neuro_cava_random:ALT_IGLV3-25|ENSG00000211659.2|EH38E3470175_fwd_tile1-1_IGLV3-25|ENSG00000211659.2|EH38E3470175|22-22639049-T-C


'cardiac_neuro_cava_random:ALT_IGLV3-25|ENSG00000211659.2|EH38E3470175_fwd_tile1-1_IGLV3-25|ENSG00000211659.2|EH38E3470175|22-22639049-T-C'

In [ ]:
# format: `<label> : <sequence_type (all ALT)> _ <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info> _ <gene name> | <ensembl id> | <enhancer/encode id> | <variant info>`
# 1: pre_header column: header without the label
# 2: "additional_gene_association" column: split the pre_header column by "~" and take the second element
# 3: tile info column: tile info (split "additional_gene_association" by "_" and take the last element) + remove this from the column
# 4: strand info column: strand info (split "additional_gene_association" by "_" and take again the last element) + remove this from the column
# 5: remove the "additional_gene_association" column from the pre_header column (split by "~" and take the first element)
# 6: pre_gene_name + ensembl_id + enhancer_id: split the pre_header column by "|" and take the first 3 elements (expand = True)
# 7: remove the pre_header column
# 8: sequence_type: put "region" if pre_gene_name does not contain "_" otherwise split the pre_gene_name column by "_" and take the first element

#### 7 pipe examples (471) 
- all alt 471
- example: 
    - cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832509~NRAS|ENSG00000213281.5|EH38E2832509_rev_tile1-1_NRAS|ENSG00000213281.5|EH38E2832509|1-114691158-A-G
    - cardiac_neuro_cava_random:ALT_DEAF1|ENSG00000177030.19|EH38E2937979~SLC25A22|ENSG00000177542.11|EH38E2937979_rev_tile1-1_SLC25A22|ENSG00000177542.11|EH38E2937979|11-745581-A-G
- "~" differenciates same region for different gene
- "_" differenciates same region for region and variant id
- variant is indicated by chr-pos-ref-alt

- format: `<label> : <sequence_type (all ALT)> _ <gene name> | <ensembl id> | <enhancer/encode id> ~ <gene name (additional gene association) | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info> _ <gene name> | <ensembl id> | <enhancer/encode id> | <variant info>`

In [25]:
num_pipes = 7
# show me an example with num_pipes pipes in the header
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes]['header'].tolist()[23]
# 'cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832509~NRAS|ENSG00000213281.5|EH38E2832509_rev_tile1-1_NRAS|ENSG00000213281.5|EH38E2832509|1-114691158-A-G'
# 'cardiac_neuro_cava_random:ALT_DEAF1|ENSG00000177030.19|EH38E2937979~SLC25A22|ENSG00000177542.11|EH38E2937979_rev_tile1-1_SLC25A22|ENSG00000177542.11|EH38E2937979|11-745581-A-G'


# # investigate and find pattern
card_cava_7_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes] 
# # does all of these rows have ALT_ after the first ":" in the header?
# card_cava_7_pipes['header'].str.split(':').str[1].str.startswith('ALT_').value_counts() # 471
# # card_cava_7_pipes['header'].str.split(':').str[1].str.startswith('REF_').value_counts() # no ref

# does each row has one ~
print("Check if all have one ~:", sum(card_cava_7_pipes['header'].str.count('~') == 1)) # 471 all have one ~ between 2 and third "|"
print("Check if all have one ~ between second and third '|': ", sum(card_cava_7_pipes['header'].str.split('\|').str[2].str.count('~') == 1)) # 471 all have one ~ between 2 and third "|"

# how many contain "rev" and how many contain "fwd"
print("All headers with rev: ", sum(card_cava_7_pipes['header'].str.count('rev'))) # 392



Check if all have one ~: 471
Check if all have one ~ between second and third '|':  471
All headers:  392


<>:3: SyntaxWarning: invalid escape sequence '\|'
<>:9: SyntaxWarning: invalid escape sequence '\|'
<>:16: SyntaxWarning: invalid escape sequence '\|'
<>:3: SyntaxWarning: invalid escape sequence '\|'
<>:9: SyntaxWarning: invalid escape sequence '\|'
<>:16: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_18448/828154450.py:3: SyntaxWarning: invalid escape sequence '\|'
  cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes]['header'].tolist()[23]
/tmp/ipykernel_18448/828154450.py:9: SyntaxWarning: invalid escape sequence '\|'
  card_cava_7_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes]
/tmp/ipykernel_18448/828154450.py:16: SyntaxWarning: invalid escape sequence '\|'
  print("Check if all have one ~ between second and third '|': ", sum(card_cava_7_pipes['header'].str.split('\|').str[2].str.count('~') == 1)) # 471 all have one ~ between 2 and third "|"


#### 8 pipe example 596
- 596 alt

In [16]:
num_pipes = 8
# show me an example with num_pipes pipes in the header
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes]['header'].tolist()[0]
# 'cardiac_neuro_cava_random:ALT_DRD4|ENSG00000069696.7|EH38E2937745_fwd_tile1-1_DEAF1|ENSG00000177030.19|EH38E2937745|11-596480-T-C~DRD4|ENSG00000069696.7|EH38E2937745|11-596480-T-C'

# investigate and find pattern
card_cava_8_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes] 
# does all of these rows have ALT_ after the first ":" in the header?
card_cava_8_pipes['header'].str.split(':').str[1].str.startswith('ALT_').value_counts() # 596
# card_cava_8_pipes['header'].str.split(':').str[1].str.startswith('REF_').value_counts() # no ref

header
True    596
Name: count, dtype: int64

#### 10 pipes in example 309
- 309 alt

In [17]:
num_pipes = 10
# show me an example with num_pipes pipes in the header
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes]['header'].tolist()[0]
# 'cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1_CSDE1|ENSG00000009307.17|EH38E2832494|1-114668983-A-G~NRAS|ENSG00000213281.5|EH38E2832494|1-114668983-A-G'

# investigate and find pattern
card_cava_10_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count('\|') == num_pipes] 
# does all of these rows have ALT_ after the first ":" in the header?
card_cava_10_pipes['header'].str.split(':').str[1].str.startswith('ALT_').value_counts() # 309
# card_cava_10_pipes['header'].str.split(':').str[1].str.startswith('REF_').value_counts() # no ref

header
True    309
Name: count, dtype: int64

### Conclusion:

cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778534_fwd_tile1-1
cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1_CSDE1|ENSG00000009307.17|EH38E2832494|1-114668983-A-G~NRAS|ENSG00000213281.5|EH38E2832494|1-114668983-A-G
cardiac_neuro_cava_random:ALT_DRD4|ENSG00000069696.7|EH38E2937745_fwd_tile1-1_DEAF1|ENSG00000177030.19|EH38E2937745|11-596480-T-C~DRD4|ENSG00000069696.7|EH38E2937745|11-596480-T-C
cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1
cardiac_neuro_cava_random:CSDE1|ENSG00000009307.17|EH38E1378377~NRAS|ENSG00000213281.5|EH38E1378377_rev_tile1-1

<label>:[ALT_/REF_]*<gene_name>|<ensembl_id>|<enhancer/encodeID_tile id>[][~<gene_name>|<ensembl_id>|<enhancer/encodeID_tile id>]*_<fwd/rev>_<tile_info>

### Prepare a tsv of all labels and get the number of sequences

In [18]:
# write sequences with same label in one file
# get list of all labels in dataframe
labels = design_df['label'].unique().tolist()
group_list_output_dir = config['general']['group_lists_directory']

for label_of_interest in labels: 
    # get the rows of the dataframe with the label of interest
    label_of_interest_df = design_df[design_df['label'] == label_of_interest]
    # write it to directory as tsv file
    label_of_interest_df.to_csv(os.path.join(group_list_output_dir, label_of_interest + f'_{len(label_of_interest_df)}.tsv'), sep='\t', index=False)


In [19]:
# get number of all rows with a label starting with "C_"
C_labels = design_df[design_df['label'].str.startswith('C_')]
C_labels

,header,sequence,label
74811,C_positive_heart_CAD:REF_rs17114036,AGGACCGGATCAACTAGGAAGCAGGTCATAATTAGTGATAGTCATT...,C_positive_heart_CAD
74812,C_positive_heart_CAD:REF_rs72664324,AGGACCGGATCAACTTCCTCTGCTGAACCCACAGCAATGGCAGCCG...,C_positive_heart_CAD
74813,C_positive_heart_CAD:REF_rs12740374,AGGACCGGATCAACTTGACCCAAAAGTGCTTCATTTTTCGTGCCCG...,C_positive_heart_CAD
74814,C_positive_heart_CAD:REF_rs4450010,AGGACCGGATCAACTTGAGGTCCAAGGATGTGAGAGTGACCACAGT...,C_positive_heart_CAD
74815,C_positive_heart_CAD:REF_rs34091558,AGGACCGGATCAACTCTTCTCGGCCAATGAAGGGTCAACTCCATTG...,C_positive_heart_CAD
...,...,...,...
77723,C_SLEA:SLEA_hg18:chr9:82902419-82902586|6:V_Rx...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGGGCCGTGACCCCGT...,C_SLEA
77724,C_SLEA:SLEA_hg18:chr9:82902419-82902586|7:V_AH...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGCGGGGATCGCGTGC...,C_SLEA
77725,C_SLEA:SLEA_hg18:chr9:82902419-82902586|80:V_H...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGCCAGGCAAGAAGTG...,C_SLEA
77726,C_SLEA:SLEA_hg18:chr9:82902419-82902586|8:V_HN...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGCCAAGGTCCAGGTG...,C_SLEA
